# Tool Calling Fine-Tuning: LoRA (16-bit) on Tesla T4 GPU
Bu notebook, **Qwen2.5-0.5B** modeli üzerinde standart **LoRA (Low-Rank Adaptation, 16-bit unquantized)** yöntemiyle Tool / Function Calling yeteneğini eğitmek ve değerlendirmek için optimize edilmiştir.

- **Metot:** LoRA (r=16, alpha=32, target_modules=[q_proj, k_proj, v_proj, o_proj])
- **Taban Model:** Qwen/Qwen2.5-0.5B (16-bit float16)
- **Hedef Donanım:** Google Colab Tesla T4 (~15GB VRAM, float16)
- **Sekans Uzunluğu:** max_seq_len = 2048 (Sistem şeması ve format yönergeleri korunur)
- **Tahmini VRAM Kullanımı:** ~4.0 - 4.5 GB (OOM riski yoktur)

---

### 1. GPU ve Donanım Kontrolü

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "HATA: GPU runtime bulunamadı!\n"
        "Lütfen Google Colab menüsünden: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU seçin!"
    )

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"bf16 support: {torch.cuda.is_bf16_supported()} (T4 için False olması normaldir; model ve eğitim float16 olarak çalışacaktır)")

### 2. Projeyi Klonla ve Çalışma Dizinine Geç

In [ ]:
import os

REPO_URL = "https://github.com/fatihkadim/tool-calling-ft.git"
PROJECT_DIR = "/content/tool-calling-ft"

# Google Colab ortamında idempotent klonlama ve çalışma dizinine geçiş
if os.path.exists("/content"):
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    !git pull origin main
else:
    print("Yerel/Özel çalışma ortamı:", os.getcwd())

!pwd

### 3. Bağımlılıkların Kurulumu ve Ortam Hazırlığı

In [ ]:
import sys
import os

# src dizinini Python import arama yoluna ekle
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

!pip install -q --upgrade pip
!pip install -q "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" "accelerate>=1.0" pyyaml tqdm pandas matplotlib

# uv_build backend'ini kur ve projeyi editable modda bağlamayı dene
!pip install -q "uv_build>=0.11.7,<0.12.0" 2>/dev/null || true
!pip install -q --no-build-isolation -e . 2>/dev/null || echo "Editable install atlandı, PYTHONPATH=src kullanılacak."

### 4. Veri Setini Kontrol Et ve Hazırla

In [ ]:
import os

train_path = "data/processed/train.jsonl"
eval_path = "data/processed/eval_subset.jsonl"

if not os.path.exists(train_path) or not os.path.exists(eval_path):
    print("İşlenmiş veri seti bulunamadı. HuggingFace'ten indirilip ChatML formatında hazırlanıyor...")
    !PYTHONPATH=src python -m tool_calling_ft.data.prepare_dataset
else:
    print(f"Veri seti hazır: {train_path} ve {eval_path} mevcut!")

### 5. T4 İçin LoRA Yapılandırması Oluştur

> **Donanım Parametreleri:** LoRA 16-bit ağırlıklarla çalışır (kuantizasyon yok).
> Tesla T4 üzerinde bellek taşması (OOM) yaşamamak için `batch_size: 2`, `grad_accum_steps: 4` (efektif batch = 8) ve `max_seq_len: 2048` kullanılır.

In [ ]:
import os
import yaml

os.makedirs("configs", exist_ok=True)

config = {
    "method": "lora",
    "base_model": "Qwen/Qwen2.5-0.5B",
    "dataset": "NousResearch/hermes-function-calling-v1",
    "output_dir": "checkpoints/lora",
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "training": {
        "epochs": 3,
        "batch_size": 2,
        "grad_accum_steps": 4,
        "learning_rate": 2e-4,
        "max_seq_len": 2048,
        "warmup_ratio": 0.05,
        "save_steps": 200,
        "seed": 42,
    },
}

config_path = "configs/lora_t4.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"T4 LoRA konfigürasyonu kaydedildi: {config_path}")
print(yaml.dump(config, default_flow_style=False))

### 6. LoRA Eğitimini Başlat

In [ ]:
# T4 GPU üzerinde 16-bit LoRA eğitimi:
# - expandable_segments:True bellek parçalanmasını önler
# - PYTHONPATH=src modül erişimini garanti eder
!PYTHONPATH=src PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.training.train --config configs/lora_t4.yaml

### 7. Canlı Demo (Inference Testi - Pozitif & Negatif Örnekler)

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = "checkpoints/lora"

print("Model ve LoRA adaptörü yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>
For each function call return a json object with function name and arguments within <tool_call> </tool_call> tags."""

test_cases = [
    ("Pozitif Örnek (Tool Çağrısı Beklenir):", "What is the weather in Tokyo in celsius?"),
    ("Negatif Örnek (Genel Sohbet / Tool Çağrılmamalı):", "What is the capital of France and what is it famous for?"),
]

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id != tokenizer.eos_token_id:
    stop_ids.append(im_end_id)

print("=" * 60)
for label, query in test_cases:
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=stop_ids,
        )

    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    print(f"\n{label}")
    print(f"Soru: {query}")
    print(f"Model Yanıtı:\n{response.strip()}")
    print("-" * 60)
print("=" * 60)

# Evaluation adımı için VRAM'i tamamen serbest bırak
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Demo tamamlandı, bellek (VRAM) temizlendi!")

### 8. LoRA Değerlendirme (Evaluation) & Metrik Tablosu

In [ ]:
import os
import json
import pandas as pd

# LoRA Modelini Değerlendir:
!PYTHONPATH=src CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.eval.harness \
    --method lora \
    --adapter checkpoints/lora \
    --dataset data/processed/eval_subset.jsonl

# Sonuç JSON raporunu yükle ve görselleştir
report_path = "reports/lora_metrics.json"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)

    print("\n" + "=" * 55)
    print(" KALİTE METRİKLERİ (QUALITY METRICS)")
    print("=" * 55)
    df_quality = pd.DataFrame(list(report.get("quality_metrics", {}).items()), columns=["Metrik", "Değer"])
    display(df_quality)

    print("\n" + "=" * 55)
    print(" PERFORMANS & DONANIM METRİKLERİ")
    print("=" * 55)
    df_perf = pd.DataFrame(list(report.get("performance_metrics", {}).items()), columns=["Metrik", "Değer"])
    display(df_perf)
else:
    print("Uyarı: reports/lora_metrics.json bulunamadı.")

### 9. Sonuçları Yedekle (Google Drive veya ZIP İndirme)

In [ ]:
import os

# 1. Seçenek: Google Drive'a Yedekleme
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dest = "/content/drive/MyDrive/tool_calling_lora_results"
    os.makedirs(drive_dest, exist_ok=True)
    !cp -r checkpoints/lora {drive_dest}/ 2>/dev/null || echo 'checkpoints klasörü kopyalanamadı'
    !cp -r reports {drive_dest}/ 2>/dev/null || echo 'reports klasörü kopyalanamadı'
    print(f"LoRA Checkpoint ve raporlar Drive'a başarıyla kaydedildi -> {drive_dest}")
except Exception as e:
    print("Google Drive bağlantısı atlandı veya hata oluştu:", e)

# 2. Seçenek: Arşiv oluşturup doğrudan bilgisayara indirme imkanı
!zip -q -r lora_results.zip checkpoints/lora reports 2>/dev/null || true
if os.path.exists("lora_results.zip"):
    print("\nlora_results.zip arşivi oluşturuldu! Sol paneldeki 'Files' sekmesinden indirebilirsiniz.")